# Report MS COCO classification challenge

**IAV — INSA Lyon**  
Tayeb and Paul

Each image can contain several of the 80 MS COCO categories. The leaderboard score is the F1 score: precision and recall weighted by the inverse class frequency. Training uses ImageNet weights only, on the 65,000 labeled images provided for the course. The official test set (4,952 images) is reserved for the submission JSON.

The experiment notebooks are still running. This file is the start of the report: how the comparison is designed, which architectures are in it, and how they will be ranked. Server-F1 numbers are left empty until each run has written `outputs/notebooks/<model>/generalization.csv`.


## Contents

1. [Use of AI](#use-of-ai)
2. [Scientific approach](#scientific-approach)
3. [Ranking of architectures](#ranking-of-architectures)
4. [Comparison diagrams](#comparison-diagrams)
5. [Bibliography](#bibliography)


<a id="use-of-ai"></a>

## 1. Use of AI

An AI coding assistant (Cursor) was used while building the project and drafting this report.

**What the assistant produced**

- A first draft of the shared library `src/coco_mlc`: datasets, the stratified split, the losses, a copy of the server metric, the training loop, and the threshold calibration.
- One notebook per architecture under `notebooks/experiments/`, from a single protocol.
- The structure and the wording of the sections below.

**What we decided, and what we check**

- The protocol in section 2: the 70/15/15 split, asymmetric loss, the two training stages, and per-class thresholds fitted on validation only.
- The shortlist in section 3, including the course baselines (VGG16, ResNet18) and the heavier models that fit on the GPU.
- Every score that will appear in the ranking. Those scores will be copied from the experiment outputs and checked against the printed metrics. They will not be estimated by the assistant.
- The submission constraints: ImageNet initialization only. Torchvision MS COCO detection weights are not used. Official test labels are not available and are not used to choose a model or a threshold.

Generated code is kept only after it has been read. A reported number comes from a full run (`FULL_TRAIN = True`). A short run (512 images, one epoch) only checks that the notebook executes.


<a id="scientific-approach"></a>

## 2. Scientific approach

### 2.1 Question

Which ImageNet-pretrained network, trained with the same transfer-learning protocol, reaches the highest server F1 on images it has not been tuned on?

The comparison is between architectures. The loss, the split, the augmentation schedule, the optimizer, and the number of epochs stay the same. Batch size is the setting that changes, because it is limited by GPU memory. It is reported with each model and counted as part of the cost.

### 2.2 Why the metric changes the objective

The evaluation server weights each class by the inverse of its frequency, then takes the harmonic mean of the weighted precision and the weighted recall. On the 65,000 labeled images, rare classes carry most of the score: `hair drier` (102 images) is about 13.6% of the score, `toaster` (117) about 11.9%, and the ten rarest classes about 43.8%. `person` (35,494 images) is about 0.04%.

Accuracy, or a micro-averaged F1, would be dominated by frequent classes and would miss the leaderboard. Training follows from that fact: a loss that does not let the many negative labels drown the rare positives, and one decision threshold per class, fitted on validation.

Training curves plot the error `100 × (1 − server F1)` at a fixed threshold of 0.5, so runs can be compared before calibration.

### 2.3 Data

| Set | Role | Size |
| --- | --- | --- |
| Labeled images | the only images with class files | 65,000 |
| Train | weight updates | 70% |
| Validation | reading the curves, threshold calibration, model selection | 15% |
| Local test | generalization, read once after the protocol is frozen | 15% |
| Official test | leaderboard JSON, no labels on our side | 4,952 |

The three labeled subsets are disjoint. The split is iterative stratification (Sechidis et al., 2011) with seed 42, cached in `outputs/split_three_stratified_0.7_0.15_0.15_42.json`, and shared by every architecture. A plain random split can leave a rare class out of validation, which makes both the metric and the thresholds unstable.

Images already have a long side of 224 and varying aspect ratios. We pad to a square, then resize to 224×224. A center crop would drop border objects whose label would still be positive. Pixels are normalized with the ImageNet mean and standard deviation, matching the pretrained backbones.

### 2.4 Training protocol

Every notebook follows the same two stages.

**Stage A — frozen backbone.** ImageNet weights, classification layer replaced by 80 logits. The backbone is frozen. Only the new head is trained, for 5 epochs, AdamW, learning rate `1e-3`, no weight decay, no dropout. Augmentation is a horizontal flip. The head bias is initialized to the log-odds of the class prevalence on the **train** subset, so the network does not start from probability 0.5 on labels that occur a few percent of the time.

**Stage B — fine-tuning.** The backbone is unfrozen. 8 epochs, AdamW, learning rate `1e-4`, weight decay `1e-4`, dropout 0.3 on the head. Augmentation adds a light affine transform (rotation, translation, scale) and color jitter. The learning rate follows a cosine schedule over the mini-batches of the stage. Mixed precision is used on CUDA.

The loss is the asymmetric loss (Ben-Baruch et al., 2021): stronger focusing on negatives (`gamma_neg = 4`, `gamma_pos = 0`) and a clip of 0.05 on easy negatives. With only a few positive labels out of 80, a plain binary cross-entropy spends most of its gradient on negatives.

The network returns logits. The sigmoid is applied only when scores become metrics or a JSON file.

### 2.5 Thresholds and the local test

After fine-tuning, probabilities are collected on the validation set. One threshold per class is chosen by coordinate ascent on the server F1, on a grid from 0.02 to 0.90. A single threshold of 0.5 is a weak default: rare classes need a lower threshold to gain recall, and that recall is heavily weighted.

The local test is scored after this choice, once, at threshold 0.5 and with the calibrated thresholds. It is not used to change the loss, the epochs, the learning rates, or the thresholds. If the local test is much worse than validation, the validation set was overfit, and we do not retune on the local test.

The official test is used only to write `submissions/exp_<model>.json`, with the thresholds chosen on validation.

### 2.6 How a model will be kept

1. Keep a run only if both stages finished on the full labeled set.
2. Rank finished runs by **calibrated server F1 on the local test**.
3. Read the calibrated validation F1 as a consistency check. The seed is shared, so this is not a second test set.
4. Break a near-tie with cost: parameter count, GFLOPs at 224×224, and the batch size that fit in memory.
5. Submit one JSON, under one group name, for the retained model.

Bias and variance are read from the train/validation error gap at the last epoch, with the cases already coded in the experiment notebooks: underfitting, overfitting, both, or a jump from validation to the local test.


<a id="ranking-of-architectures"></a>

## 3. Ranking of architectures

### 3.1 Shortlist

Eleven networks, all from torchvision, initialized with `Weights.DEFAULT` (ImageNet). They cover the families seen in the course and the families that fit on one GPU. MobileNetV3-Small, EfficientNet-B1 and EfficientNet-B3 sit in the library registry and are outside this comparison.

| Family | Why it is in the shortlist |
| --- | --- |
| VGG16, ResNet18 | Course baselines. VGG is deep and expensive. ResNet18 is the residual reference at modest cost. |
| ShuffleNet V2, MobileNetV3-Large, EfficientNet-B0 | Small models. They test whether a cheap backbone is enough once the head and the thresholds target rare classes. |
| ResNet50, EfficientNet-B4 | Mid-size convnets with a higher ImageNet accuracy. |
| ConvNeXt-Tiny, Swin-T, MaxViT-T, EfficientNetV2-S | Recent convnet and transformer designs, around 20–30 M parameters. |

The figures below come from the torchvision weights table, at 224×224, before our head replacement. They are published reference numbers. They are not MS COCO scores. ResNet50 uses the `IMAGENET1K_V2` weights.

### 3.2 Reference table, ordered by compute

This order is a cost order. The challenge ranking is section 3.3.

| Architecture | Params (M) | GFLOPs | ImageNet top-1 (%) | Batch size in our runs |
| --- | ---: | ---: | ---: | ---: |
| ShuffleNet V2 ×1.0 | 2.3 | 0.14 | 69.36 | 128 |
| MobileNetV3-Large | 5.5 | 0.22 | 75.27 | 96 |
| EfficientNet-B0 | 5.3 | 0.39 | 77.69 | 64 |
| ResNet18 | 11.7 | 1.81 | 69.76 | 128 |
| ResNet50 | 25.6 | 4.09 | 80.86 | 96 |
| EfficientNet-B4 | 19.3 | 4.39 | 83.38 | 16 |
| ConvNeXt-Tiny | 28.6 | 4.46 | 82.52 | 64 |
| Swin-T | 28.3 | 4.49 | 81.47 | 64 |
| MaxViT-T | 30.9 | 5.56 | 83.70 | 8 |
| EfficientNetV2-S | 21.5 | 8.37 | 84.23 | 16 |
| VGG16 | 138.4 | 15.47 | 71.59 | 8 |

EfficientNetV2-S has the highest ImageNet top-1 of the shortlist and one of the highest compute costs. VGG16 has the most parameters, a batch of 8, and a lower ImageNet accuracy than several models ten times smaller. ImageNet top-1 is a prior: the server F1 weights rare classes, so a strong ImageNet model can still miss `hair drier` or `toaster`.

### 3.3 Challenge ranking

Primary key: calibrated server F1 on the local test. The table stays empty until the runs finish.

| Rank | Architecture | Val F1 @ 0.5 | Val F1 calibrated | Local-test F1 @ 0.5 | Local-test F1 calibrated | Diagnosis |
| --- | --- | --- | --- | --- | --- | --- |
| — | *pending* | | | | | |

Source, once each notebook has finished: `outputs/notebooks/<architecture>/generalization.csv` and `diagnosis.json`.


<a id="comparison-diagrams"></a>

## 4. Comparison diagrams

### 4.1 What stays fixed, and what changes

```mermaid
flowchart LR
  subgraph fixed [Same for every architecture]
    S[Seed 42<br/>stratified 70 / 15 / 15]
    L[Asymmetric loss]
    A[Stage A: frozen head<br/>5 epochs, lr 1e-3]
    B[Stage B: fine-tune<br/>8 epochs, lr 1e-4]
    T[Thresholds on validation]
  end
  subgraph varies [Changes]
    M[Backbone]
    G[Batch size]
  end
  S --> A --> B --> T
  M --> A
  G --> A
  T --> R[Local test, read once]
  R --> J[One leaderboard JSON]
```

### 4.2 Families on the shortlist

```mermaid
flowchart TB
  subgraph course [Course baselines]
    VGG[VGG16]
    R18[ResNet18]
  end
  subgraph light [Light CNNs]
    SH[ShuffleNet V2]
    MV[MobileNetV3-Large]
    EB0[EfficientNet-B0]
  end
  subgraph mid [Mid-size CNNs]
    R50[ResNet50]
    EB4[EfficientNet-B4]
    EV2[EfficientNetV2-S]
  end
  subgraph recent [Recent backbones]
    CN[ConvNeXt-Tiny]
    SW[Swin-T]
    MX[MaxViT-T]
  end
```

### 4.3 From the labeled set to the submission

```mermaid
flowchart TD
  D[65,000 labeled images] --> SP[Iterative stratification, seed 42]
  SP --> TR[Train 70%]
  SP --> VA[Validation 15%]
  SP --> LO[Local test 15%]
  TR --> H[Train the head, backbone frozen]
  H --> F[Fine-tune the full network]
  F --> C[Calibrate one threshold per class on validation]
  VA --> C
  C --> LO
  LO --> PICK[Keep the best calibrated local-test F1]
  PICK --> OFF[Official test, 4,952 images, no labels]
  OFF --> JSON[JSON of class indices]
```

### 4.4 Where the server score comes from

These shares are a property of the labeled set. They do not depend on a network.

| Group | Share of the server score | What it implies |
| --- | --- | --- |
| Ten rarest classes | about 43.8% | Recall on these classes moves the leaderboard. |
| `hair drier` alone | about 13.6% | One missed rare positive costs more than many errors on frequent classes. |
| `toaster` alone | about 11.9% | Same mechanism, second weight. |
| `person` | about 0.04% | A high accuracy on people barely changes the score. |

### 4.5 Head and backbone

```mermaid
flowchart LR
  IMG[Image 224 x 224<br/>pad, ImageNet norm] --> BB[Backbone<br/>ImageNet weights]
  BB --> FEAT[Feature vector]
  FEAT --> HEAD[Linear head<br/>80 logits]
  HEAD --> SIG[Sigmoid, at evaluation only]
  SIG --> DEC[Per-class thresholds<br/>fit on validation]
```

Stage A freezes the backbone and trains the head. Stage B trains both, with a smaller learning rate and dropout on the head.

### 4.6 Published cost, side by side

The next cell draws the three reference axes (parameters, GFLOPs, ImageNet top-1). It does not read any training log. The MS COCO comparison chart will be added in section 3.3 once the runs exist.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Published torchvision figures at 224x224, plus the batch size used in our notebooks.
# This is a reference comparison, not a MS COCO ranking.
rows = [
    ("ShuffleNet V2", 2.3, 0.14, 69.36, 128),
    ("MobileNetV3-L", 5.5, 0.22, 75.27, 96),
    ("EfficientNet-B0", 5.3, 0.39, 77.69, 64),
    ("ResNet18", 11.7, 1.81, 69.76, 128),
    ("ResNet50", 25.6, 4.09, 80.86, 96),
    ("EfficientNet-B4", 19.3, 4.39, 83.38, 16),
    ("ConvNeXt-T", 28.6, 4.46, 82.52, 64),
    ("Swin-T", 28.3, 4.49, 81.47, 64),
    ("MaxViT-T", 30.9, 5.56, 83.70, 8),
    ("EfficientNetV2-S", 21.5, 8.37, 84.23, 16),
    ("VGG16", 138.4, 15.47, 71.59, 8),
]
names = [r[0] for r in rows]
params = np.array([r[1] for r in rows])
gflops = np.array([r[2] for r in rows])
top1 = np.array([r[3] for r in rows])
order = np.argsort(gflops)
names = [names[i] for i in order]
params, gflops, top1 = params[order], gflops[order], top1[order]
y = np.arange(len(names))

fig, axes = plt.subplots(1, 3, figsize=(12.5, 5.2), sharey=True)
panels = [
    (axes[0], params, "Parameters (M)", "#3d5a80"),
    (axes[1], gflops, "GFLOPs at 224 px", "#ee6c4d"),
    (axes[2], top1, "ImageNet top-1 (%)", "#2a9d8f"),
]
for ax, values, title, color in panels:
    ax.barh(y, values, color=color)
    ax.set_title(title)
    ax.grid(axis="x", linestyle=":", alpha=0.6)
axes[0].set_yticks(y, names)
axes[0].set_xlabel("millions")
axes[1].set_xlabel("GFLOPs")
axes[2].set_xlabel("%")
axes[2].set_xlim(65, 88)
fig.suptitle("Reference comparison — published cost, not the MS COCO score", fontsize=12)
fig.tight_layout()

out = Path("outputs") / "report"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "architecture_reference.png", dpi=140, bbox_inches="tight")
plt.show()


<a id="bibliography"></a>

## 5. Bibliography

**Assignment and data**

- T.-Y. Lin, M. Maire, S. Belongie, J. Hays, P. Perona, D. Ramanan, P. Dollár, and C. L. Zitnick. *Microsoft COCO: Common Objects in Context.* ECCV, 2014.
- R. Kéchichian. *The MS COCO classification challenge.* Course notebook, IAV, INSA Lyon. Defines the metric, the dataset layout, and the ban on any other MS COCO distribution.
- Torchvision models and weights. <https://docs.pytorch.org/vision/main/models.html>. Source of the parameter counts, GFLOPs, and ImageNet top-1 figures in section 3.

**Architectures**

- K. Simonyan and A. Zisserman. *Very Deep Convolutional Networks for Large-Scale Image Recognition.* ICLR, 2015.
- K. He, X. Zhang, S. Ren, and J. Sun. *Deep Residual Learning for Image Recognition.* CVPR, 2016.
- N. Ma, X. Zhang, H.-T. Zheng, and J. Sun. *ShuffleNet V2: Practical Guidelines for Efficient CNN Architecture Design.* ECCV, 2018.
- A. Howard, M. Sandler, G. Chu, et al. *Searching for MobileNetV3.* ICCV, 2019.
- M. Tan and Q. V. Le. *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML, 2019.
- M. Tan and Q. V. Le. *EfficientNetV2: Smaller Models and Faster Training.* ICML, 2021.
- Z. Liu, Y. Lin, Y. Cao, et al. *Swin Transformer: Hierarchical Vision Transformer using Shifted Windows.* ICCV, 2021.
- Z. Liu, H. Mao, C.-Y. Wu, C. Feichtenhofer, T. Darrell, and S. Xie. *A ConvNet for the 2020s.* CVPR, 2022.
- Z. Tu, H. Talebi, H. Zhang, et al. *MaxViT: Multi-Axis Vision Transformer.* ECCV, 2022.
- O. Russakovsky, J. Deng, H. Su, et al. *ImageNet Large Scale Visual Recognition Challenge.* International Journal of Computer Vision, 2015.

**Method**

- J. Yosinski, J. Clune, Y. Bengio, and H. Lipson. *How transferable are features in deep neural networks?* NeurIPS, 2014.
- T.-Y. Lin, P. Goyal, R. Girshick, K. He, and P. Dollár. *Focal Loss for Dense Object Detection.* ICCV, 2017. The head bias is initialized from the class prevalence, following the same idea.
- E. Ben-Baruch, T. Ridnik, N. Zamir, et al. *Asymmetric Loss For Multi-Label Classification.* ICCV, 2021.
- K. Sechidis, G. Tsoumakas, and I. Vlahavas. *On the Stratification of Multi-label Data.* ECML PKDD, 2011.
- I. Loshchilov and F. Hutter. *Decoupled Weight Decay Regularization.* ICLR, 2019.
